# Libraries and environment

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
from datetime import date
import json
import re

In [ ]:
MXSCR=10
UPDATE=False

In [ ]:
today = date.today()

# Building of service files and updated versions of models and listings files

In [ ]:
#previous version of models file
if UPDATE:
    links_listingsold=pd.read_json('/kaggle/input/testmodels0-1/model_links_listingsold.json')

In [ ]:
#new version of models file
links_listings=pd.read_json('/kaggle/input/testmodels0-1/model_links_listings.json')

## Creation of new version urls and reference numbers files

In [ ]:
urls_listnew=[]
for jj in list(pd.DataFrame(links_listings['listing_urls']).index):
    urls_listnew.extend(pd.DataFrame(links_listings['listing_urls']).loc[jj].values[-1])
pd.DataFrame(urls_listnew).to_csv('urls_listnew.csv')

In [ ]:
refnum_list=list(links_listings.refnum.unique())
pd.DataFrame(refnum_list).to_csv('refnum_list.csv')

## Creation of log dictionary for all old dropped listings urls

In [ ]:
listingsmod=pd.read_json('/kaggle/input/listings1/listings')['Basic Info'].apply(pd.Series)

In [ ]:
listingsmod.isna().sum()

In [ ]:
urls_listold=list(pd.read_json('/kaggle/input/listings1/listings')['URL'].apply(pd.Series)['url'])

In [ ]:
urls_listnew_=[('https://www.chrono24.com'+y.strip()) for y in urls_listnew]

In [ ]:
urls_drop=[x for x in urls_listold if x not in urls_listnew_]

In [ ]:
dicurls_drop={}
for tt in urls_drop:
  dicurls_drop[tt]=today
json.dump(dicurls_drop,open("urls_dropped.json",'w' ))

## Creation of log dictionary for all old dropped reference numbers

In [ ]:
refnum_drop=[]
if UPDATE:
  refnum_drop=[x for x in list(links_listingsold.refnum.unique()) if x not in refnum_list]

In [ ]:
refnum_listnewadd=[]
if UPDATE:
    refnum_listnewadd=[y for y in refnum_list if y not in list(links_listingsold.refnum.unique())]

In [ ]:
if UPDATE:
  dicrefnum_drop=json.load(open(f'/kaggle/input/dicrefnum_drop/dicrefnum_drop.json'))
  for kk in refnum_drop:
   if kk in dicrefnum_drop.keys():
    if kk in refnum_listnewadd:
      dicrefnum_drop[kk][today]='used'
    else:
      dicrefnum_drop[kk][today]='unused'
   else:
    dicrefnum_drop[kk]={today:'unused'}
else:
  dicrefnum_drop={}
  for kk in refnum_drop:
    dicrefnum_drop[kk]={today:'unused'}
with open("dicrefnum_drop.json", "w", encoding="utf-8") as file:
    file.write(json.dumps(dicrefnum_drop, indent=4))

## Building of updated version models file

In [ ]:
if UPDATE:
  refnum_drop_=[]
  for key,value in dicrefnum_drop.items():
      for key_,value_ in value.items():
          lastvalue=value_
      if lastvalue=='used':
         refnum_drop_.append(key)

In [ ]:
if UPDATE:
  listlinks_listingsold=[]
  for rr in refnum_drop_:
    listlinks_listingsold.append(links_listingsold[links_listingsold.refnum==rr])
  links_listingsoldadd=pd.concat(listlinks_listingsold,axis=0)
  links_listings1=pd.concat([links_listings,links_listingsoldadd],axis=0)
  links_listings1=links_listings1.reset_index(drop=True)

In [ ]:
#index_nodrop=list(pd.read_json('/kaggle/input/listings1/listings')['URL'].apply(pd.Series).reset_index().set_index('url').drop(urls_drop,axis=0)['index'])

In [ ]:
#if len(urls_drop)==0:
#    _=0
#else:
#  listingsmod=listingsmod.loc[index_nodrop]

## Creation of file that includes available models names for each of brands

In [ ]:
listingsmod['Brand'].unique()

In [ ]:
listingsmod['Brand'].nunique()

In [ ]:
br=list(listingsmod.groupby('Brand')['Brand'].count().sort_values(ascending=False)[0:50].index)

In [ ]:
br

In [ ]:
lst_modelsinbrand=[]
for vv in br:
     modelsinbrand=pd.DataFrame(listingsmod[listingsmod['Brand']==vv].groupby('Model')['Model'].count().sort_values(ascending=False)[0:15].index).T
     modelsinbrand['brand']=vv
     lst_modelsinbrand.append(modelsinbrand)

In [ ]:
table_of_models=pd.concat(lst_modelsinbrand,axis=0).sort_values(by='brand').fillna('Unknown')
table_of_models=pd.DataFrame(np.array(table_of_models),columns=list(table_of_models.columns.astype('string'))).set_index('brand')
table_of_models

In [ ]:
table_of_models.to_excel("table_of_models.xlsx")

## Creation of file that includes all available brands names and file that includes available brands names duplicated by the number of scripts

In [ ]:
lst_=[]
maxscript=MXSCR
for s in range(0,maxscript):
  lst_.append(pd.DataFrame(br,columns=[f'brands{s}']))
table_of_brandsinloops=pd.concat(lst_,axis=1)

In [ ]:
table_of_brandsinloops.drop(br.index('Rolex'),axis=0,inplace=True)

In [ ]:
table_of_brandsinloops.reset_index(drop=True,inplace=True)

In [ ]:
table_of_brandsinloops

In [ ]:
table_of_brandsinloops.to_excel("table_of_brandsinloops.xlsx")

In [ ]:
(pd.DataFrame(br,columns=['brands'])).to_excel("table_of_brands.xlsx")

In [ ]:
#listings=pd.read_json('/kaggle/input/listings1/listings')
#if len(urls_drop)==0:
#    _=0
#else:
#  listings=listings.loc[index_nodrop]
#for BRND in br:
#  a=0
#  listings_exm=[]
#  print(f'brand:{BRND}')
#  for jj in list(listingsmod[listingsmod.Brand==BRND].index):
#   a=a+1
#   if a%100==0:
#        print(f'file: {a}')
#   listings_exm.append(listings.loc[jj,:].to_dict())
#  BRND=re.sub('& ','',BRND)
#  BRND=re.sub('ö','',BRND)
#  BRND=re.sub('ü','',BRND)
#  ins=re.sub('è','',BRND)
#  with open(f"{ins.lower()}_listings", "w", encoding="utf-8") as file:
#      file.write(json.dumps(listings_exm, indent=4))


## Updated version models file saving

In [ ]:
if UPDATE:
  links_listings1jsn=[]
  for jj in list(links_listings1.index):
     links_listings1jsn.append(links_listings1.loc[jj,:].to_dict())
  with open("model_links_listings.json", "w", encoding="utf-8") as file:
      file.write(json.dumps(links_listings1jsn, indent=4))